# 01. 탐색적 데이터 분석 (EDA)

DILI 데이터셋의 기본 통계, 라벨 분포, 분자 특성 분포를 확인합니다.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from rdkit import Chem
from rdkit.Chem import Descriptors, Draw

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100

## 1. 데이터 로드

In [ ]:
df = pd.read_csv('../data/processed/dili_dataset.csv')
print(f'데이터 크기: {df.shape}')
print(f'\n컬럼: {list(df.columns)}')
df.head()

## 2. 라벨 분포

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# 막대 그래프
label_counts = df['Label'].value_counts().sort_index()
axes[0].bar(['Non-toxic (0)', 'Toxic (1)'], label_counts.values,
            color=['#2ecc71', '#e74c3c'])
axes[0].set_ylabel('Count')
axes[0].set_title('Label Distribution')
for i, v in enumerate(label_counts.values):
    axes[0].text(i, v + 5, str(v), ha='center', fontweight='bold')

# 파이 차트
axes[1].pie(label_counts.values, labels=['Non-toxic', 'Toxic'],
            colors=['#2ecc71', '#e74c3c'], autopct='%1.1f%%',
            startangle=90)
axes[1].set_title('Label Ratio')

plt.tight_layout()
plt.savefig('../results/figures/label_distribution.png', dpi=150)
plt.show()

print(f'\n독성(1): {label_counts.get(1, 0)}개 ({label_counts.get(1, 0)/len(df)*100:.1f}%)')
print(f'비독성(0): {label_counts.get(0, 0)}개 ({label_counts.get(0, 0)/len(df)*100:.1f}%)')

## 3. 분자 물리화학적 특성 분포

In [ ]:
# 각 분자의 물리화학적 특성 계산
desc_data = []
for smi in df['SMILES']:
    mol = Chem.MolFromSmiles(smi)
    if mol:
        desc_data.append({
            'MolWt': Descriptors.MolWt(mol),
            'MolLogP': Descriptors.MolLogP(mol),
            'TPSA': Descriptors.TPSA(mol),
            'NumHDonors': Descriptors.NumHDonors(mol),
            'NumHAcceptors': Descriptors.NumHAcceptors(mol),
            'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        })

desc_df = pd.DataFrame(desc_data)
desc_df['Label'] = df['Label'].values[:len(desc_df)]

desc_df.describe()

In [ ]:
# 독성 vs 비독성 그룹별 분포 비교
props = ['MolWt', 'MolLogP', 'TPSA', 'NumHDonors', 'NumHAcceptors', 'NumRotatableBonds']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))

for ax, prop in zip(axes.flat, props):
    for label, color, name in [(0, '#2ecc71', 'Non-toxic'), (1, '#e74c3c', 'Toxic')]:
        subset = desc_df[desc_df['Label'] == label][prop]
        ax.hist(subset, bins=30, alpha=0.6, color=color, label=name, density=True)
    ax.set_title(prop)
    ax.legend()

plt.suptitle('Physicochemical Properties: Toxic vs Non-toxic', fontsize=14)
plt.tight_layout()
plt.savefig('../results/figures/property_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. 분자 구조 시각화 (예시)

In [ ]:
# 독성/비독성 각 4개씩 시각화
toxic_samples = df[df['Label'] == 1].head(4)
nontoxic_samples = df[df['Label'] == 0].head(4)

mols_toxic = [Chem.MolFromSmiles(s) for s in toxic_samples['SMILES']]
mols_nontoxic = [Chem.MolFromSmiles(s) for s in nontoxic_samples['SMILES']]

print('Toxic compounds:')
img = Draw.MolsToGridImage(mols_toxic, molsPerRow=4, subImgSize=(300, 200))
display(img)

print('\nNon-toxic compounds:')
img = Draw.MolsToGridImage(mols_nontoxic, molsPerRow=4, subImgSize=(300, 200))
display(img)

## 5. 요약

- 데이터셋 크기와 라벨 분포 확인
- 불균형이 있다면 `class_weight='balanced'` 등으로 처리 필요
- 독성/비독성 그룹 간 물리화학적 특성 차이 관찰